In [ ]:
%load_ext autoreload
%autoreload 3 --print --log

In [ ]:
# 从项目根目录或 examples 目录启动均可；统一以项目根目录运行。
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (cwd, *cwd.parents)
     if (p / "mtp_initializer").is_dir() and (p / "PROJECT.md").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("请从 scipykit 项目根目录或 examples 目录启动 notebook")
os.chdir(PROJECT_ROOT)
# 根目录用于本地脚本；父目录用于 import scipykit。同步对子进程生效。
python_paths = [str(PROJECT_ROOT), str(PROJECT_ROOT.parent)]
for path in reversed(python_paths):
    if path not in sys.path:
        sys.path.insert(0, path)
os.environ["PYTHONPATH"] = os.pathsep.join(
    dict.fromkeys(python_paths + [p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if p])
)
os.environ["NOTEBOOK_NAME"] = "01_绘图快速上手"
print("项目路径:", PROJECT_ROOT)
print("当前解释器:", sys.executable)


In [ ]:
{**dict(a=1), **dict(a=2,b=1)}

# 科研绘图快速上手
选择已安装示例依赖的 Python 内核后从上到下运行。下面保留熟悉的星号导入方式；图像写入根目录的 `assets/01_绘图快速上手/`。

In [ ]:
from scipykit.mtp_initializer import *
# 导入已自动应用历史 rcParams 默认预设，直接开始画图。
configure_inline(("png",))
x = np.linspace(0, 2 * np.pi, 200)
fig = mfigure((6, 3), dpi=150, alpha=0)
ax = fig.add_axes([0.13, 0.19, 0.82, 0.70])
ax.plot(x, np.sin(x), color=palette()[0], label="sin(x)")
label_axes(ax, "x", "y", "Example curve", grid=True, legend=True)
disp(fig, "name")
# 沿用上一句的 name.png，透明背景、300 dpi 导出；原画布保持原样。
saved = savefig(fig, dpi=300, alpha=0)
print("已导出:", saved)
plt.close(fig)

## 多子图、误差带和论文配色
`subplots` 返回原生 Figure/Axes；元素函数返回原生 Artist，可继续定制。临时 `plot_style` 上下文退出后恢复设置。

In [ ]:
with plot_style("paper", **{"font.size": 9}):
    fig, axes = subplots(1, 2, figsize=(7, 2.8), layout="constrained")
    for i, ax in enumerate(axes):
        mean = np.sin(x + i * 0.6)
        line, band = plot_band(ax, x, mean, std=0.15 + 0.05 * np.cos(x),
                               color=palette()[i], label="mean ± std")
        label_axes(ax, "Time", "Value", grid=True, legend=True)
        panel_label(ax, f"({chr(97 + i)})", xy=(0, 1.03))
    disp(fig, "uncertainty", embed=True)
    savefig(fig, "assets/01_绘图快速上手/uncertainty.pdf", alpha=0)
    pixels = canvas_to_array(fig, mode="RGBA")
    print("像素数组:", pixels.shape, pixels.dtype)
    plt.close(fig)

## 显示和保存的约定
- `disp(fig)` 直接显示；`disp(fig, key)` 外置 PNG，并记住绝对路径；私有 MIME 同样输出绝对路径。
- `embed=True` 同时内嵌图像，分享 notebook 时不依赖外置资源。
- `savefig(fig)` 未绑定名称时使用 `assets/figure.png`，重复调用会覆盖同一路径。
- `savefig(fig, path)` 支持 PNG/PDF/SVG 等 Matplotlib 格式。
- `mfigure` 保留历史背景 Axes，`fig.axes[0]` 是背景；布局复杂时用 `subplots`。
- 坐标换算适用于二维直角轴；对数轴用 `at=(x, y)` 指定局部参考点。